# Tutorial: Palette catalog summaries

## Audience
- Analysts or developers who want quick summaries from a color catalog CSV.

## Prerequisites
- Basic Python functions, dictionaries, lists, and file paths.

## Learning goals
- Read UTF-8 CSV data by header names.
- Normalize and group rows by primary color.
- Build a sorted summary for downstream reporting.


## Outline
1. Define `summarize_palettes(csv_path)`.
2. Normalize `Primary (Hex)` and `Product Type`.
3. Count rows and collect unique product types per primary color.
4. Sort by count descending, then primary ascending.
5. Run the function on `/inputs/colors.csv`.


In [ ]:
from __future__ import annotations

import csv


def summarize_palettes(csv_path: str) -> list[dict[str, object]]:
    counts: dict[str, int] = {}
    product_type_sets: dict[str, set[str]] = {}

    with open(csv_path, mode="r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames is None:
            return []

        for row in reader:
            primary = (row.get("Primary (Hex)") or "").strip().upper()
            product_type = (row.get("Product Type") or "").strip()

            counts[primary] = counts.get(primary, 0) + 1
            product_type_sets.setdefault(primary, set()).add(product_type)

    summary = [
        {
            "primary": primary,
            "count": count,
            "product_types": sorted(product_type_sets[primary]),
        }
        for primary, count in counts.items()
    ]
    summary.sort(key=lambda item: (-item["count"], item["primary"]))
    return summary


In [ ]:
import os
import tempfile

# Schema-compatible sample with reordered rows, duplicates, lowercase colors, and a quoted comma.
with tempfile.NamedTemporaryFile("w", encoding="utf-8", newline="", delete=False) as tmp:
    sample_path = tmp.name
    tmp.write("Product Type,Primary (Hex),Notes\n")
    tmp.write("\"Home, Decor\", #aa00aa ,quoted comma\n")
    tmp.write("Poster,#AA00AA,duplicate group\n")
    tmp.write("Poster,#AA00AA,duplicate row\n")
    tmp.write("Cards,#00ff00,second group\n")

sample_summary = summarize_palettes(sample_path)
assert sample_summary == [
    {"primary": "#AA00AA", "count": 3, "product_types": ["Home, Decor", "Poster"]},
    {"primary": "#00FF00", "count": 1, "product_types": ["Cards"]},
]

# Header-only input should return an empty list.
with tempfile.NamedTemporaryFile("w", encoding="utf-8", newline="", delete=False) as tmp_header:
    header_only_path = tmp_header.name
    tmp_header.write("Product Type,Primary (Hex)\n")

assert summarize_palettes(header_only_path) == []

# Empty catalog should also return an empty list.
with tempfile.NamedTemporaryFile("w", encoding="utf-8", newline="", delete=False) as tmp_empty:
    empty_path = tmp_empty.name

assert summarize_palettes(empty_path) == []

for path in (sample_path, header_only_path, empty_path):
    os.unlink(path)


## Exercises
- Add one more row to a local CSV and predict how ordering changes.
- Try a product type with leading/trailing spaces and verify stripping.
- Explain why duplicates increase `count` but not unique `product_types`.


In [ ]:
palette_summary = summarize_palettes('/inputs/colors.csv')
